In [ ]:
import pandas as pd
import geopandas as gpd
from shapely import Point
from sklearn.pipeline import Pipeline

from src.features.geo import water_fraction, within_shape, extract_polygon
from src.preprocessing.preprocessing import construct_basin_location_mapper, col_splitter, binarizer, \
    construct_basin_location_mapper_with_external

train_oil_df = pd.read_csv("../data/train_oil.csv")
test_oil_df = pd.read_csv("../data/oil_test.csv")

In [ ]:
train_oil_df.columns = [c.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_") for c in
                        train_oil_df.columns]
train_oil_df.columns

In [ ]:
train_str_cols = train_oil_df.select_dtypes(include='str').columns
test_str_cols = test_oil_df.select_dtypes(include='str').columns
train_oil_df[train_str_cols] = train_oil_df[train_str_cols].apply(lambda x: x.str.lower())
test_oil_df[test_str_cols] = test_oil_df[test_str_cols].apply(lambda x: x.str.lower())

#### Пропущенные координаты

In [ ]:
missing_coordinates = pd.read_csv("../data/missing_coordinates.csv")
missing_coordinates = missing_coordinates.fillna("")

#### Onshore/offshore по бассейнам

In [ ]:
train_oil_df[['']]

#### Геоданные

##### Гео-датасеты

In [ ]:
land = gpd.read_file("../data/ne_50m_land.zip")['geometry']
land_10m = gpd.read_file("../data/ne_10m_land.zip")['geometry']
ocean_50m = gpd.read_file("../data/ne_50m_ocean.zip")['geometry']
ocean_110m = gpd.read_file("../data/ne_110m_ocean.zip")['geometry']
ne_50m_geography_marine_polys = gpd.read_file("../data/ne_50m_geography_marine_polys.zip")
lakes = gpd.read_file("../data/ne_50m_lakes.zip")['geometry']
geography_regions = gpd.read_file("../data/ne_50m_geography_regions_polys.zip")
rivers_lake_centerlines = gpd.read_file("../data/ne_50m_rivers_lake_centerlines.zip")['geometry']
major_islands = gpd.read_file("../data/ne_50m_coastline.zip")['geometry']
minor_islands = gpd.read_file("../data/ne_10m_minor_islands.zip")['geometry']

##### Формирование подмножеств геопризнаков

In [ ]:
bays = ne_50m_geography_marine_polys[ne_50m_geography_marine_polys.featurecla == 'bay']['geometry']
gulfs = ne_50m_geography_marine_polys[ne_50m_geography_marine_polys.featurecla == 'gulf']['geometry']
straits = ne_50m_geography_marine_polys[ne_50m_geography_marine_polys.featurecla == 'strait']['geometry']
channels = ne_50m_geography_marine_polys[ne_50m_geography_marine_polys.featurecla == 'channel']['geometry']
sounds = ne_50m_geography_marine_polys[ne_50m_geography_marine_polys.featurecla == 'sound']['geometry']
rivers = ne_50m_geography_marine_polys[ne_50m_geography_marine_polys.featurecla == 'river']['geometry']
isthmuses = geography_regions[geography_regions.FEATURECLA == 'Isthmus']['geometry']
deltas = geography_regions[geography_regions.FEATURECLA == 'Delta']['geometry']
coasts = geography_regions[geography_regions.FEATURECLA == 'Coast']['geometry']
lake_regions = geography_regions[geography_regions.FEATURECLA == 'Lake']['geometry']
islands = geography_regions[geography_regions.FEATURECLA.isin(['Island group', 'Island'])]['geometry']

In [ ]:
bunyu_coords = Point(117.833, 3.5)
bunyu_island = gpd.GeoSeries(
    [extract_polygon(land_10m[land_10m.geometry.contains(bunyu_coords)].iloc[0], bunyu_coords)],
    crs="EPSG:4326"
)

In [ ]:
all_lakes = gpd.GeoDataFrame(
    geometry=pd.concat([lakes, rivers_lake_centerlines, lake_regions], ignore_index=True),
    crs="EPSG:4326").dissolve()

In [ ]:
all_islands = gpd.GeoDataFrame(
    geometry=pd.concat([islands, major_islands, minor_islands, bunyu_island], ignore_index=True),
    crs="EPSG:4326").dissolve()

##### Бассейны

In [ ]:
basins_mapped = pd.read_csv("../data/basins_mapped.csv")

### Подготовка данных

In [ ]:
train_oil_df_filled = train_oil_df.merge(missing_coordinates[["field_name", "reservoir_unit", "latitude", "longitude"]],
                                         on=["field_name", "reservoir_unit"],
                                         how='left', suffixes=('', '_fill'))
train_oil_df_filled['latitude'] = train_oil_df_filled['latitude'].fillna(train_oil_df_filled['latitude_fill'])
train_oil_df_filled['longitude'] = train_oil_df_filled['longitude'].fillna(train_oil_df_filled['longitude_fill'])

In [ ]:
gdf = gpd.GeoDataFrame(
    train_oil_df_filled,
    geometry=gpd.points_from_xy(train_oil_df_filled.longitude, train_oil_df_filled.latitude),
    crs="EPSG:4326"
)

### Подготовка признаков

#### Процент воды в радиусе 2км

In [ ]:
gdf['water_pct'] = gdf.apply(lambda row: water_fraction(point_lat=row.latitude,
                                                        point_lon=row.longitude,
                                                        water_shapes=ocean_50m,
                                                        radius_km=2), axis=1)

#### Расположение в воде/на суше

In [ ]:
gdf['is_in_water'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=ocean_50m), axis=1)

#### Корректировка координат Berri Hanifa

In [ ]:
berri_hanifa_coords = (27.11, 49.6389)
berri_hanifa_idx = (gdf["field_name"] == "berri") & (gdf["reservoir_unit"] == "hanifa")
gdf.loc[berri_hanifa_idx, 'latitude'] = berri_hanifa_coords[0]
gdf.loc[berri_hanifa_idx, 'longitude'] = berri_hanifa_coords[1]
gdf.loc[berri_hanifa_idx, 'water_pct'] = water_fraction(point_lat=berri_hanifa_coords[0], 
                                                        point_lon=berri_hanifa_coords[1], 
                                                        water_shapes=ocean_50m,
                                                        radius_km=2)

#### Гео-признаки

In [ ]:
gdf['is_on_island'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=all_islands), axis=1)
gdf['is_in_gulf'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=gulfs), axis=1)
gdf['is_in_strait'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=straits), axis=1)
gdf['is_in_delta'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=deltas), axis=1)
gdf['is_in_bay'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=bays), axis=1)

#### Извлечение отметок onshore/offshore, lake из названий местрождения и бассейна

In [ ]:
names_concat = gdf['field_name'] + gdf['reservoir_unit'] + gdf['basin_name']
gdf['ref_onshore'] = names_concat.str.contains('onshore') 
gdf['ref_offshore'] = names_concat.str.contains('offshore')
gdf['ref_lake'] = names_concat.str.contains("lake")

#### Метки локаций бассейнов (расчётные и внешние)

In [ ]:
basin_location = pd.crosstab(gdf['basin_name'], gdf['onshore_offshore']).drop('unknown', errors='ignore').reset_index()

loc_mask = ((basin_location['onshore'] > 0) &
            (basin_location['offshore'] > 0))

basin_location.loc[loc_mask, 'onshore-offshore-calc'] = (
        basin_location.loc[loc_mask, 'onshore'] +
        basin_location.loc[loc_mask, 'offshore'] +
        basin_location.loc[loc_mask, 'onshore-offshore'])

basin_location['onshore-offshore-calc'] = basin_location['onshore-offshore-calc'].fillna(0).astype(int)

loc_mask_2 = ((basin_location['onshore'] == 0) &
              (basin_location['offshore'] == 0) &
              (basin_location['onshore-offshore'] > 0))

basin_location.loc[loc_mask_2, 'onshore-offshore-calc'] = basin_location.loc[loc_mask_2, 'onshore-offshore']
inferred_location_map = construct_basin_location_mapper(basin_location)
gdf['basin_location_inferred'] = gdf['basin_name'].map(inferred_location_map).fillna("unknown")

In [ ]:
basins_mapped = basins_mapped.drop(basins_mapped[basins_mapped['basin_name'] == 'unknown'].index).reset_index(drop=True).to_dict("records")
external_location_map = {b['basin_name']: b['location'] for b in basins_mapped}
basin_location_map_combined = construct_basin_location_mapper_with_external(basin_location, external_location_map)
gdf['basin_location_external'] = gdf['basin_name'].map(basin_location_map_combined).fillna('unknown')

### Разбивка и кодировка композитных полей 

### tectonic_regime

In [ ]:
tectonic_regime_binarizer = Pipeline([
    ('extract', col_splitter(col='tectonic_regime')),
    ('binarize', binarizer())
])

tectonic_regime_binarized = tectonic_regime_binarizer.fit_transform(train_oil_df)

pd.DataFrame(tectonic_regime_binarized,
             columns=[f"tectonic_regime_{c.replace(' ', '-')}" for c in
                      tectonic_regime_binarizer.named_steps['binarize'].func.__self__.classes_.tolist()],
             index=train_oil_df.index)

### structural_setting

In [ ]:
structural_setting_binarizer = Pipeline([
    ('extract', col_splitter(col='structural_setting')),
    ('binarize', binarizer())
])

structural_setting_binarized = structural_setting_binarizer.fit_transform(train_oil_df)

pd.DataFrame(structural_setting_binarized,
             columns=[f"structural_setting_{c.replace(' ', '-')}" for c in
                      structural_setting_binarizer.named_steps['binarize'].func.__self__.classes_.tolist()],
             index=train_oil_df.index)